# Chapter 8: Calculation of Molecular Properties
## 8.3. Molecular orbital analysis

Molecular orbitals connect an electronic-structure calculation to a useful picture of bonding. We will compare the conjugated π systems of **1,3-butadiene and 1,3,5-hexatriene** using a small Hückel model, then examine actual **RHF/STO-3G orbitals of H₂**. Keeping the models small lets us inspect every step.

### Learning objectives

By the end, you should be able to:

- distinguish a basis function, a molecular orbital, and a many-electron wavefunction;
- identify occupied orbitals, the HOMO, the LUMO, and their energy difference;
- check MO normalization in a nonorthogonal atomic-orbital basis;
- interpret orbital phase, nodes, and density without treating an orbital gap as an optical transition;
- explain what a simple conjugation model can and cannot predict.

**Setup:** use the activated course Conda environment from the [README](Readme.md). Run this notebook independently from top to bottom. It performs one two-electron SCF calculation and small matrix/grid operations, uses one Psi4 thread and 512 MiB, and requires no downloaded geometry, cube file, or browser viewer.

### Start here: reading orbital pictures without memorizing shapes

An **atomic orbital (AO) basis function** is a building block centered near an atom. A **molecular orbital (MO)** is a weighted combination of such functions across the molecule. Its coefficients describe an amplitude with a sign (or, more generally, a phase). Opposite colors in an orbital picture indicate opposite signs, not positive and negative electrical charge.

Electrons occupy orbitals according to the chosen electronic model. **HOMO** means highest occupied MO; **LUMO** means lowest unoccupied MO. Orbital energies help organize bonding and possible excitations, but an orbital-energy difference is not automatically an observed absorption energy.

In a planar conjugated carbon chain, the σ framework connects nuclei along the bond directions. The π system comes from side-by-side overlap of p orbitals perpendicular to the molecular plane. A conjugated chain lets those π amplitudes extend over several neighboring atoms.

**First pass:** use the small π-electron chain model to see how combinations produce nodes, energy levels, and bonding patterns. Then compare with an actual H₂ SCF calculation. The overlap-matrix identities explain why coefficients from the two models cannot be interpreted in exactly the same way.

In [ ]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import psi4

output_dir = Path("outputs/chapter08_part3")
output_dir.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print(f"NumPy {np.__version__}; Psi4 {psi4.__version__}")

### 8.3.1. From basis functions to molecular orbitals

An MO is a one-electron function expanded in a chosen basis:

$$
\psi_i(\mathbf r)=\sum_\mu C_{\mu i}\chi_\mu(\mathbf r).
$$

The coefficients describe the contribution **and relative phase** of each basis function. They are not generally probabilities assigned to atoms: atom-centered basis functions overlap. With overlap matrix $S_{\mu\nu}=\langle\chi_\mu|\chi_\nu\rangle$, orthonormal MOs satisfy $C^\mathsf{T}SC=I$ for real coefficients.

In closed-shell RHF, each occupied spatial MO contains two electrons of opposite spin. The many-electron approximation is an antisymmetric Slater determinant built from the corresponding spin orbitals. An individual MO is neither an electron trajectory nor the many-electron wavefunction.

A canonical MO solves a one-electron eigenvalue problem. Its eigenvalue $\epsilon_i$ is an **orbital energy**, distinct from the molecule's total energy. In HF the occupied orbital energies cannot simply be summed to get the total energy, because that would count electron–electron interactions incorrectly.

### 8.3.2. A transparent model of conjugation

Simple Hückel theory keeps one perpendicular p orbital per conjugated carbon, assumes an orthonormal site basis $S=I$, and uses a nearest-neighbor Hamiltonian:

$$
H_{ij}=\begin{cases}
\alpha & i=j,\\
\beta & |i-j|=1,\\
0 & \text{otherwise}.
\end{cases}
$$

Here $\alpha=0$ defines the energy zero and $\beta=-1$ defines the scale. The eigenvalues are therefore expressed in **units of $|\beta|$**, not eV. Each neutral carbon contributes one π electron; a chain of four or six carbons has two or three doubly occupied π MOs.

This is a qualitative π-electron model with equal couplings. It omits the σ framework, explicit electron correlation, bond-length alternation, and the dependence of p-orbital coupling on torsion. It is not an ab initio calculation or a quantitative prediction of an absorption wavelength. See the [IUPAC definition of Hückel theory](https://goldbook.iupac.org/terms/view/HT07035).

In [ ]:
def huckel_chain(n_sites):
    if n_sites < 2 or n_sites % 2:
        raise ValueError("Use an even number of sites for this neutral closed-shell example.")
    hamiltonian = -np.eye(n_sites, k=1) - np.eye(n_sites, k=-1)
    energies, coefficients = np.linalg.eigh(hamiltonian)
    # A global sign is arbitrary. Use a reproducible display convention.
    for column in range(n_sites):
        if coefficients[0, column] < 0:
            coefficients[:, column] *= -1
    occupations = np.zeros(n_sites, dtype=int)
    occupations[:n_sites // 2] = 2
    np.testing.assert_allclose(coefficients.T @ coefficients, np.eye(n_sites), atol=1e-12)
    np.testing.assert_allclose(hamiltonian @ coefficients, coefficients * energies, atol=1e-12)
    return energies, coefficients, occupations

chains = {"Butadiene": huckel_chain(4), "Hexatriene": huckel_chain(6)}
rows = []
for name, (energies, coefficients, occupations) in chains.items():
    n_occ = np.count_nonzero(occupations)
    rows.append({"π system": name, "π electrons": int(occupations.sum()),
                 "HOMO / |β|": energies[n_occ - 1], "LUMO / |β|": energies[n_occ],
                 "gap / |β|": energies[n_occ] - energies[n_occ - 1]})
pd.DataFrame(rows).round(6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharey=True, layout="constrained")
for ax, (name, (energies, coefficients, occupations)) in zip(axes, chains.items()):
    n_occ = np.count_nonzero(occupations)
    for index, (energy, occupancy) in enumerate(zip(energies, occupations)):
        color = "#245e8f" if occupancy else "#b33b53"
        ax.hlines(energy, 0.15, 0.70, color=color, linewidth=2.5)
        label = f"π{index + 1}, {occupancy} e⁻"
        if index == n_occ - 1:
            label += "  HOMO"
        elif index == n_occ:
            label += "  LUMO"
        ax.text(0.74, energy, label, va="center", fontsize=10)
    ax.annotate("", xy=(0.07, energies[n_occ]), xytext=(0.07, energies[n_occ - 1]),
                arrowprops={"arrowstyle": "<->", "color": "black"})
    ax.set(xlim=(0, 1.65), xticks=[], title=name, xlabel="Simple Hückel π orbitals")
    ax.axhline(0, color="0.8", linewidth=0.7)
axes[0].set_ylabel("Orbital energy / |β| (α = 0)")
plt.show()

For a chain of $N$ identical sites, the analytic result is

$
\epsilon_k=\alpha+2\beta\cos\!\left(\frac{k\pi}{N+1}\right),\quad k=1,\ldots,N.
$

With negative $\beta$, this orders the levels from low to high. The calculated gaps are approximately **1.236 $|\beta|$** for butadiene and **0.890 $|\beta|$** for hexatriene. Longer conjugation reduces the gap **within this model**. Real polyenes also change geometry and electronic structure; this model alone cannot establish a universal gap law for molecules.

In [ ]:
for name, (energies, coefficients, occupations) in chains.items():
    n_sites = len(energies)
    analytic = -2 * np.cos(np.arange(1, n_sites + 1) * np.pi / (n_sites + 1))
    np.testing.assert_allclose(energies, analytic, atol=1e-12)
print("Matrix eigenvalues agree with the analytic chain solution.")

### 8.3.3. Phase and nodes: drawing the frontier π orbitals

The following is a **lobe diagram**, not a spatial isosurface. Each site has an upper and lower p lobe of opposite sign. Circle area is proportional to $|C_{\mu i}|^2$; color gives the sign of $C_{\mu i}$ in the upper lobe. Site positions are schematic.

All these π orbitals have the molecular plane as a nodal plane. The extra sign changes **along the chain** help distinguish the frontier orbitals. Changing the overall sign of an entire MO changes neither its density nor its physics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 5.8), layout="constrained")
positive, negative = "#bc3956", "#24699c"
for column, (name, (energies, coefficients, occupations)) in enumerate(chains.items()):
    n_occ = np.count_nonzero(occupations)
    sites = np.arange(1, len(energies) + 1)
    for row, (orbital, label) in enumerate([(n_occ - 1, "HOMO"), (n_occ, "LUMO")]):
        ax = axes[row, column]
        c = coefficients[:, orbital]
        for side in [-1, 1]:
            colors = np.where(side * c > 0, positive, negative)
            ax.scatter(sites, np.full(len(sites), side * 0.43), s=1800 * c**2,
                       c=colors, edgecolors="white", linewidth=0.7)
        ax.plot(sites, np.zeros(len(sites)), "o-", color="0.55", markersize=4)
        nodes = int(np.count_nonzero(c[:-1] * c[1:] < 0))
        ax.set(xlim=(0.4, len(sites) + 0.6), ylim=(-0.9, 0.95), xticks=sites, yticks=[],
               xlabel="Carbon site", title=f"{name} {label}: {nodes} chain sign changes")
axes[0, 0].legend(handles=[Patch(color=positive, label="positive phase"),
                          Patch(color=negative, label="negative phase")],
                  loc="upper left", fontsize=8, ncol=2)
plt.show()

### Worked research question: are all neighboring π bonds equivalent?

Conjugated molecules often have different bond lengths along a chain. Can the simplest delocalized-electron model already distinguish neighboring bonds even when every coupling $\beta$ was set equal?

In this **orthonormal Hückel π model**, form the occupied electron-density matrix

$$P_{ij}^{\pi}=\sum_k n_k C_{ik}C_{jk},$$

where $n_k$ is 2 for an occupied orbital and 0 otherwise. Its diagonal counts π electrons assigned to each site; a nearest-neighbor off-diagonal element is the model's **π bond order**. It is dimensionless, and it is not an integer formal bond order or a general AO-basis bond-index definition. This is the Coulson bond-order convention discussed in [Hosoya's paper on electron density and bond order](https://publications.iupac.org/pac/pdf/1983/pdf/5502x0269.pdf).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4.2), layout="constrained")
pi_bond_rows = []
for ax, (name, (_, coefficients, occupations)) in zip(axes, chains.items()):
    pi_density = (coefficients * occupations) @ coefficients.T
    n_sites = len(occupations)
    np.testing.assert_allclose(np.diag(pi_density), 1.0, atol=1e-12)
    assert np.isclose(np.trace(pi_density), occupations.sum())
    for i in range(n_sites - 1):
        order = float(pi_density[i, i+1])
        ax.plot([i, i+1], [0, 0], lw=2 + 7*order, color="#28788e", solid_capstyle="round")
        ax.text(i + 0.5, 0.14, f"{order:.3f}", ha="center", fontsize=10)
        pi_bond_rows.append({"chain": name, "bond": f"{i+1}–{i+2}", "pi_bond_order": order})
    ax.scatter(np.arange(n_sites), np.zeros(n_sites), s=280, color="white", edgecolor="black", zorder=3)
    for i in range(n_sites):
        ax.text(i, 0, str(i+1), ha="center", va="center", zorder=4, fontsize=9)
    ax.set(title=name + ": bond widths show π bond order", xlim=(-0.4, n_sites-0.6), ylim=(-0.3, 0.45))
    ax.axis("off")
pi_bond_table = pd.DataFrame(pi_bond_rows)
pi_bond_table.to_csv(output_dir / "huckel_pi_bond_orders.csv", index=False)
fig.savefig(output_dir / "huckel_bond_order_pattern.png", dpi=140)
plt.show()
pi_bond_table.round(4)

**Conclusion.** Equal Hamiltonian couplings do not force equal occupied π bond orders: filling the delocalized orbitals produces an alternating pattern. This is a useful qualitative hypothesis about where electronic bonding differs along the chain.

**Next step:** optimize the molecular geometry with a method that includes the full σ and π framework, then compare bond lengths and a stated bond-index definition. These dimensionless numbers cannot be converted directly to ångströms. **Self-check:** flipping an entire occupied MO's sign leaves $C_{ik}C_{jk}$ unchanged, so this bonding conclusion cannot depend on an arbitrary orbital color swap.

### 8.3.4. Actual canonical orbitals: RHF/STO-3G H₂

We now use two hydrogen nuclei separated by **0.74 Å**, charge 0 and singlet multiplicity 1. This is a fixed geometry. The two contracted STO-3G basis functions produce two spatial MOs: occupied bonding $\sigma_g$ and unoccupied antibonding $\sigma_u^*$.

The minimal basis is useful for inspecting coefficients, but is too limited for quantitative spectroscopy. We also avoid stretching H₂ into the bond-breaking region, where the single closed-shell RHF determinant becomes qualitatively inadequate.

Psi4 is reset explicitly so calculations in other notebooks cannot silently change this model. Coordinates stay in the supplied orientation. All software output goes to the notebook's own output folder.

In [ ]:
psi4.core.clean()
psi4.core.clean_options()
psi4.set_num_threads(1)
psi4.set_memory("512 MiB")
scratch_dir = (output_dir / "scratch").resolve()
scratch_dir.mkdir(exist_ok=True)
psi4.core.IOManager.shared_object().set_default_path(str(scratch_dir))
psi4.set_output_file(str(output_dir / "h2_rhf.log"), False)
psi4.set_options({"basis": "sto-3g", "reference": "rhf", "scf_type": "pk", "guess": "sad",
                  "e_convergence": 1e-10, "d_convergence": 1e-10, "maxiter": 50,
                  "fail_on_maxiter": True})
h2 = psi4.geometry("""0 1
H -0.37 0.0 0.0
H  0.37 0.0 0.0
units angstrom
symmetry c1
no_reorient
no_com
""")
start = perf_counter()
total_energy, wfn = psi4.energy("hf", molecule=h2, return_wfn=True)
print(f"RHF/STO-3G total energy: {total_energy:.10f} Eh")
print(f"SCF wall time: {perf_counter() - start:.3f} s")

In [ ]:
# C1 symmetry makes these ordinary AO matrices, without symmetry blocks.
C = wfn.Ca().np.copy()        # rows: basis functions; columns: spatial MOs
S = psi4.core.MintsHelper(wfn.basisset()).ao_overlap().np.copy()
orbital_energies = wfn.epsilon_a().np.copy()
n_occ = wfn.nalpha()
assert n_occ == wfn.nbeta() == 1 and C.shape == (2, 2)
assert np.all(np.isfinite(orbital_energies)) and np.all(np.diff(orbital_energies) > 0)
np.testing.assert_allclose(C.T @ S @ C, np.eye(2), atol=1e-9)
# Fix the arbitrary phase for consistent figures and tables.
for i in range(C.shape[1]):
    if C[0, i] < 0:
        C[:, i] *= -1
hartree_to_ev = psi4.constants.hartree2ev
homo, lumo = n_occ - 1, n_occ
print("AO overlap matrix S:\n", S)
print("MO coefficient matrix C:\n", C)
print(f"HOMO–LUMO gap: {(orbital_energies[lumo] - orbital_energies[homo]) * hartree_to_ev:.4f} eV")
pd.DataFrame({"MO": ["σg (HOMO)", "σu* (LUMO)"], "electrons": [2, 0],
              "energy / Eh": orbital_energies,
              "energy / eV": orbital_energies * hartree_to_ev}).round(6)

Because the two normalized AOs overlap by $s=S_{12}$, the normalized combinations are

$$
\psi_g=\frac{\chi_A+\chi_B}{\sqrt{2(1+s)}},\qquad
\psi_u=\frac{\chi_A-\chi_B}{\sqrt{2(1-s)}}.
$$

The antibonding coefficients can exceed 1: their values compensate for cancellation between overlapping functions. This does not mean that an atom has a probability greater than 1. The normalization condition is $C^\mathsf{T}SC=I$, not $C^\mathsf{T}C=I$.

**What does the gap mean?** It is a difference between two eigenvalues of this RHF Fock operator. It is neither the total energy change on excitation nor an absorption wavelength. Optical excitations involve an interacting electron and hole, and can involve orbital relaxation and correlation. Within frozen-orbital HF, Koopmans' approximation relates $-\epsilon_\mathrm{HOMO}$ to an ionization energy; it does not turn every orbital-energy difference into a measured transition. Kohn–Sham gaps have their own functional-dependent interpretation. Part 8.2 instead used differences between appropriate total energies.

In [ ]:
s = S[0, 1]
analytic_C = np.array([[1 / np.sqrt(2 * (1 + s)), 1 / np.sqrt(2 * (1 - s))],
                       [1 / np.sqrt(2 * (1 + s)), -1 / np.sqrt(2 * (1 - s))]])
np.testing.assert_allclose(C, analytic_C, atol=1e-9)
print("Computed coefficients agree with the overlap-normalized bonding/antibonding combinations.")

### 8.3.5. A small spatial slice of the calculated orbitals

A cube file stores values on a three-dimensional grid. For this diatomic molecule, a **two-dimensional slice through the bond axis** already exposes the bonding and antibonding patterns.

We evaluate the actual contracted basis functions used by Psi4, then form $\psi_i=\sum_\mu\chi_\mu C_{\mu i}$. The point evaluator takes coordinates in **bohr**, so we convert the plotted Å coordinates before evaluation. Its local-to-global index map identifies the basis functions that survive numerical screening. The array named `PHI` contains AO basis values, not MO values.

In [ ]:
def orbital_values(points_angstrom, basis, coefficients):
    points_bohr = np.asarray(points_angstrom, dtype=float) / psi4.constants.bohr2angstroms
    if points_bohr.ndim != 2 or points_bohr.shape[1] != 3:
        raise ValueError("Coordinates must have shape (number of points, 3).")
    vectors = [psi4.core.Vector.from_array(points_bohr[:, axis].copy()) for axis in range(3)]
    weights = psi4.core.Vector.from_array(np.ones(len(points_bohr)))
    extents = psi4.core.BasisExtents(basis, 1e-12)
    block = psi4.core.BlockOPoints(*vectors, weights, extents)
    evaluator = psi4.core.BasisFunctions(basis, len(points_bohr), basis.nbf())
    evaluator.compute_functions(block)
    local_to_global = np.array(block.functions_local_to_global(), dtype=int)
    phi = evaluator.basis_values()["PHI"].np[:len(points_bohr), :len(local_to_global)]
    return (phi @ coefficients[local_to_global, :]).copy()

x = np.linspace(-2.5, 2.5, 121)
z = np.linspace(-2.0, 2.0, 91)
X, Z = np.meshgrid(x, z)
points = np.column_stack([X.ravel(), np.zeros(X.size), Z.ravel()])
mo_slice = orbital_values(points, wfn.basisset(), C).reshape(*X.shape, 2)
assert np.all(np.isfinite(mo_slice))
print(f"Evaluated {X.size:,} points × {C.shape[1]} MOs; no cube file is needed.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True, layout="constrained")
limit = np.max(np.abs(mo_slice))
for orbital, (ax, title) in enumerate(zip(axes, ["Occupied σg (HOMO)", "Virtual σu* (LUMO)"])):
    values = mo_slice[:, :, orbital]
    artist = ax.contourf(X, Z, values, levels=np.linspace(-limit, limit, 25), cmap="RdBu_r")
    if orbital == lumo:
        ax.contour(X, Z, values, levels=[0], colors="black", linewidths=1)
    ax.scatter([-0.37, 0.37], [0, 0], s=24, color="black", edgecolors="white", zorder=3)
    ax.set(xlabel="x (Å)", title=title, aspect="equal")
axes[0].set_ylabel("z (Å), y = 0")
fig.colorbar(artist, ax=axes, label=r"Signed MO amplitude ($a_0^{-3/2}$)", shrink=0.85)
plt.show()

The bonding MO has the same phase across both nuclei. The antibonding MO changes sign at the plane halfway between them; the black line is that plane's intersection with this slice. Red and blue indicate opposite phases, **not positive and negative charge**.

For this RHF state, the total electron density is $\rho(\mathbf r)=2|\psi_g(\mathbf r)|^2$; the unoccupied MO contributes no ground-state electron density. A three-dimensional density integrates to two electrons, but integrating a single plane does **not** give the electron count. Nor does a colored lobe enclose the entire orbital: the function extends beyond any selected contour.

In [ ]:
density_slice = 2 * mo_slice[:, :, homo]**2
flipped = orbital_values(points, wfn.basisset(), -C).reshape(*X.shape, 2)
np.testing.assert_allclose(density_slice, 2 * flipped[:, :, homo]**2, atol=1e-12)
# The antibonding amplitude vanishes everywhere on the midpoint plane x = 0.
midpoint_plane = np.column_stack([np.zeros(len(z)), np.zeros(len(z)), z])
midpoint_values = orbital_values(midpoint_plane, wfn.basisset(), C)
assert np.max(np.abs(midpoint_values[:, lumo])) < 1e-10
print("Global phase reversal leaves density unchanged; the antibonding midpoint node is verified.")
psi4.core.clean()

### Exercises

1. In the Hückel code, use eight sites. How many occupied π MOs are there, and what happens to the frontier gap? Check using the analytic formula.
2. Why does doubling $|\beta|$ double every Hückel gap but leave the eigenvectors unchanged? Why is choosing an arbitrary $\beta$ in eV insufficient to predict an absorption spectrum?
3. A classmate normalizes each column of `C` with its Euclidean length. Explain why this breaks the H₂ calculation's normalization.
4. If every coefficient in the antibonding MO changes sign, does its node disappear? Does its energy change?
5. Why can the unoccupied H₂ MO be visualized even though it contributes no electrons to the RHF ground-state density?
6. Which additional calculation would be appropriate for an optical excitation: subtracting the printed orbital energies, or an excited-state method with a suitable basis and accuracy assessment?

<details>
<summary>Suggested answers</summary>

1. Four doubly occupied π MOs. The gap is $4\sin[\pi/(2(8+1))]\approx0.6946$ in $|\beta|$ units, smaller than for six sites.
2. With $\alpha=0$, multiplying the Hamiltonian by a positive scalar scales its eigenvalues and preserves its eigenvectors. The simple model omits important geometry and interacting-electron physics; a fitted parameter is not an excitation calculation.
3. Nonorthogonal AOs require the overlap metric: each orbital satisfies $\mathbf c^\mathsf{T}S\mathbf c=1$. The ordinary squared coefficients omit overlap cross terms.
4. Neither. Relative phases, the nodal plane, and all observables remain unchanged by a global sign.
5. The Fock operator also has virtual eigenfunctions. They form part of the available one-electron space, even when their occupation is zero.
6. A suitable excited-state method, such as linear-response TDDFT or an appropriate wavefunction method, followed by basis/method checks. A computed excitation also needs transition information to interpret spectral intensity.

</details>

### References and next step

- [IUPAC: Hückel molecular orbital theory](https://goldbook.iupac.org/terms/view/HT07035) — definition and model scope.
- [Psi4: Hartree–Fock theory](https://psicode.org/psi4manual/master/scf.html) — SCF equations, references, and orbital output.
- [Psi4 Wavefunction API](https://psicode.org/psi4manual/master/api/psi4.core.Wavefunction.html) — coefficient matrices, orbital energies, and occupation counts.
- [Psi4 BasisFunctions API](https://psicode.org/psi4manual/master/api/psi4.core.BasisFunctions.html) and [BlockOPoints API](https://psicode.org/psi4manual/master/api/psi4.core.BlockOPoints.html) — evaluation of the actual basis on a small user-defined grid.
- [Psi4 time-dependent SCF](https://psicode.org/psi4manual/master/tdscf.html) — an example of calculating excitation energies and transition properties explicitly.

Next: [Part 8.4 — Potential energy surfaces](Chapter08_Part4.ipynb), where the coordinates change and we compare total model energies.